# NAS共同来源分钟特征读通

使用原有20个完成会话特征，不新增因子、不拟合、不回测。核查来源哈希、Qlib读取、缺失掩码、下一交易日可用时点。历史补采不等于实盘首收认证。

In [1]:
from pathlib import Path
import os
root=Path.cwd()
while not (root/"SPEC.md").exists(): root=root.parent
os.chdir(root)
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
card=root/"research/nas_minute_features_20261014"


In [2]:
import json, hashlib
import numpy as np
import pandas as pd
import exchange_calendars as xc
import qlib
from qlib.workflow import R
from src.data.nas_minute_bundle import load_nas_minute_bundle
from src.data.minute_allocation_features import completed_session_features
from src.data.qlib_native import QlibProviderConfig, build_qlib_store_from_frame, load_qlib_feature_frame
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
plan=json.loads((card/'plan.json').read_text())
binding=json.loads((card/'nas_binding.json').read_text())
assert binding['source_id']==plan['source_id']
raw,events=load_nas_minute_bundle(binding,workspace_root=root)
provider=root/plan['provider'];native={};manifests={}
fields=['open','high','low','close','volume','price_valid']
for key,symbols,freq in [('core',['SOXL','SOXX'],'1min'),('peer',['QQQ','SMH','NVDA','AMD'],'5min')]:
    rows=raw.loc[raw.symbol.isin(symbols)].rename(columns={'symbol':'instrument','timestamp':'datetime'})
    manifests[key]=build_qlib_store_from_frame(rows,provider_uri=provider/key,freq=freq,as_of=plan['as_of'],field_map={f:f for f in fields},source='Pinned NAS raw NONE '+plan['source_id'])
    qlib.init(provider_uri=str(provider/key),region='us')
    native[key]=load_qlib_feature_frame(QlibProviderConfig(provider/key,freq=freq),instruments=symbols,fields=['$'+f for f in fields],start=plan['start'],end=plan['as_of']+' 16:00:00')
source_counts=raw.groupby('symbol').size().to_dict();del raw,rows
schedule=xc.get_calendar('XNYS',start=plan['start'],end=plan['as_of']).schedule
features,coverage=completed_session_features(native['core'],native['peer'],schedule,core_names=plan['feature_groups']['core'],peer_names=plan['feature_groups']['peer'],peer_symbols=['QQQ','SMH','NVDA','AMD'])
numeric=[*plan['feature_groups']['core'],*plan['feature_groups']['peer'],'minute_core_ready','minute_peer_ready']
rows=features[numeric].reset_index();rows['instrument']='SOXL_SOXX_CASH'
manifests['features']=build_qlib_store_from_frame(rows,provider_uri=provider/'features',freq='day',as_of=plan['as_of'],field_map={f:f for f in numeric},source='Completed-session raw NONE features from NAS '+plan['source_id'])
observed=load_qlib_feature_frame(QlibProviderConfig(provider/'features'),instruments=['SOXL_SOXX_CASH'],fields=['$'+f for f in numeric],start=plan['start'],end=plan['as_of']).xs('SOXL_SOXX_CASH',level='instrument').rename(columns=lambda c:c.removeprefix('$'))
assert observed.index.equals(features.index)
np.testing.assert_allclose(observed.to_numpy(),features[numeric].to_numpy(),rtol=1e-6,atol=1e-6,equal_nan=True)
# Trade-date mapping uses the real exchange calendar, including next Oct14 inference date.
next_schedule=xc.get_calendar('XNYS',start=plan['start'],end='2026-10-14').schedule
next_days=pd.Series(next_schedule.index,index=next_schedule.index).shift(-1).reindex(features.index)
timing=pd.DataFrame({'feature_available_at':features.minute_feature_available_at,'trade_date':next_days},index=features.index)
timing['decision_at']=pd.to_datetime(timing.trade_date).dt.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=29)
assert timing.decision_at.notna().all() and (timing.feature_available_at<timing.decision_at).all()
assert observed.loc['2026-10-12':,['minute_core_ready','minute_peer_ready']].eq(1).all().all()
assert observed.loc['2020-03-16',numeric].isna().sum()==20
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
summary={'experiment':plan['experiment'],'source_id':plan['source_id'],'as_of':plan['as_of'],'region':'us','provider_uri':str(provider/'features'),'start':plan['start'],'feature_sessions':len(features),'feature_count':20,'new_sessions':2,'source_rows':source_counts,'ready_core_sessions':int(observed.minute_core_ready.sum()),'ready_peer_sessions':int(observed.minute_peer_ready.sum()),'latest_feature_session':str(features.index.max().date()),'latest_decision_at':str(timing.decision_at.iloc[-1]),'historical_first_receipt_certified':False,'all_model_inputs_rebuilt':False,'production_unchanged':True,'new_fits':0,'new_backtests':0,'objective_achieved':False,'decision':'minute_features_connected_daily_inputs_and_model_not_refit','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='nas_minute_features',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(source_id=plan['source_id'],new_fits=0,new_backtests=0)
    rec.save_objects(artifact_path='inputs',plan=plan,binding=binding,provider_manifests=manifests)
    rec.save_objects(artifact_path='analysis',features=observed,timing=timing,coverage=coverage,summary=summary)
    rec.save_objects(artifact_path='sources',loader=(root/'src/data/nas_minute_bundle.py').read_text(),features=(root/'src/data/minute_allocation_features.py').read_text())
    artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts'
    rec.log_metrics(feature_sessions=len(features),registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking),plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest())
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
atomic_json(summary,card/'summary.json')
features.loc['2026-10-12':].to_csv(card/'latest_features.csv')
coverage.loc[~coverage.core_ready | ~coverage.peer_price_ready].to_csv(card/'unready_sessions.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':str(provider/'features'),'recorder_uri':str(tracking),'source_id':plan['source_id'],'region':'us','as_of':plan['as_of'],'artifact_policy':summary['artifact_policy']},sort_keys=False))
print({k:summary[k] for k in ['feature_sessions','ready_core_sessions','ready_peer_sessions','latest_decision_at','recorder_id']})


# NAS分钟特征已接通，尚未形成新超额证据

固定来源 `2c70bd56ffc15ad4ba4eb243e0ca0b9409cd9d39f369c0d95ac02829de1fcf58`，按原有12个核心路径与8个同行特征重新构建Qlib数据，未增加因子、拟合模型或回测。输入为SOXL/SOXX各780840条1分钟记录，以及QQQ/SMH/NVDA/AMD各156168条5分钟记录；使用raw NONE会话内比率，不引入未来复权系数。

覆盖2018-10-12至2026-10-13，共2010个交易日。2009日核心和同行均就绪，新增10月12、13日的20个特征均非缺失；2020-03-16因原始OHLC异常，保留当天并保持20个数值缺失、两个可用标志为0。未删除压力日、未填造行情。完整历史包括半日市，按实际收盘构建。

原入口只能识别旧采集清单。现在通过NAS binding校验源清单、上下文指针及对象哈希，读取核心与同行；不同来源时区统一为纽约时间，保留原接收时间。现有allocator准备函数已接入新loader，旧来源格式继续兼容。规范化源写入独立Qlib provider，再经D.features读取、计算并写回日频特征provider；读取结果与构建结果一致。

所有完成会话特征的假定可用时间均早于下一实际交易日09:29。10月13日数据对应2026-10-14 09:29纽约决策。此为历史可用时点合同，不是历史首收或独立前向认证；新两日也不能用于验证稳定超额。

决定：**minute_features_connected_daily_inputs_and_model_not_refit**。0拟合、0回测、未晋升，现役release哈希未变。完整模型还需要日频/PIT输入与标签时钟核对，`all_model_inputs_rebuilt=false`；本卡不宣称新的收益或风险改善。

2项相关检查通过；已执行notebook，NAS Recorder冷恢复并校验12项对象。Recorder `140b7394874849a1b068241f8f120ff2`。原始数据权威仍为NAS，完整特征、时点表和覆盖表保留在Recorder；Git只保留小型两日特征摘要及异常日摘要。

下一步核对现役39个日频输入与统一来源的关系，补齐必要输入后进行冻结模型推理校验；已失败的分钟特征研究路线不因更新两天数据而重新调参。

[新增两日特征](latest_features.csv) · [保留的异常日](unready_sessions.csv) · [验证清单](verification_manifest.json)
